<a href="https://colab.research.google.com/github/Ritesh-panda/datascience-learnings/blob/main/FE_KPCA_QDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.feature_selection import SequentialFeatureSelector as SFS

In [2]:
df=pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.head(5
        )

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"Missing values in TotalCharges after conversion: {df['TotalCharges'].isnull().sum()}")
display(df[['TotalCharges']].info())


Missing values in TotalCharges after conversion: 11
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 1 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   TotalCharges  7032 non-null   float64
dtypes: float64(1)
memory usage: 55.2 KB


None

In [6]:
df.dropna(subset=['TotalCharges'], inplace=True)
print(f'Rows remaining: {len(df)}')
display(df.isnull().sum())

Rows remaining: 7032


,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [7]:
df.Churn.value_counts()/len(df)*100

,count
Churn,
No,73.421502
Yes,26.578498


In [8]:
x=df.drop(['customerID','Churn'],axis=1)


y=df.Churn.values

In [9]:
# Identify categorical columns (excluding numeric ones)
categorical_cols = x.select_dtypes(include=['object']).columns.tolist()

# Apply dummy encoding
x = pd.get_dummies(x, columns=categorical_cols, drop_first=True)

print("Encoded features shape:", x.shape)
display(x.head())

Encoded features shape: (7032, 30)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,False,True,False,True,False


In [10]:
#training and testing data
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3, random_state= 42)

In [11]:
#build the model
model1=LogisticRegression()
model1.fit(x_train,y_train)
y_pred=model1.predict(x_test)
y_pred

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


array(['No', 'No', 'Yes', ..., 'No', 'No', 'No'], dtype=object)

In [12]:
print("Accuracy:",accuracy_score(y_test,y_pred))

Accuracy: 0.7924170616113744


**APPLY PCA**

In [13]:
from sklearn.decomposition import PCA
pca=PCA(n_components=15)
x_train_pca=pca.fit_transform(x_train)
x_test_pca=pca.transform(x_test)


In [14]:
model2=LogisticRegression()
model2.fit(x_train_pca,y_train)
y_pred_pca=model2.predict(x_test_pca)
y_pred_pca

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


array(['No', 'No', 'Yes', ..., 'No', 'No', 'No'], dtype=object)

In [15]:
accuracy_score(y_test,y_pred_pca)

0.7962085308056872

**APPLY LDA**

In [16]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
LDA=LDA()
x_train_LDA=LDA.fit_transform(x_train,y_train)
x_test_LDA=LDA.transform(x_test)


In [22]:
model3=LogisticRegression()
model3.fit(x_train_LDA,y_train)
y_pred_LDA=model1.predict(x_test_LDA)
y_pred_LDA
accuracy_score(y_test,y_pred_LDA)


0.795260663507109

KPCA

In [18]:
from sklearn.decomposition import KernelPCA as KPCA
Kpca=KPCA(n_components=15)
x_train_Kpca=Kpca.fit_transform(x_train)
x_test_Kpca=Kpca.transform(x_test)


In [19]:
model4=LogisticRegression()
model4.fit(x_train_Kpca,y_train)
y_pred_Kpca=model4.predict(x_test_Kpca)
y_pred_Kpca

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


array(['No', 'No', 'Yes', ..., 'No', 'No', 'No'], dtype=object)

In [20]:
accuracy_score(y_test,y_pred_Kpca)


0.7976303317535545

QDA

In [30]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
QDA=QDA()
x_train_QDA=QDA.fit(x_train,y_train)
y_pred_QDA=QDA.predict(x_test)


/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


In [31]:

accuracy_score(y_test,y_pred_QDA)

0.7028436018957346